# GridWatch: Chronos-2 (zero-shot and fine-tuned) on Colab

The foundation-model arm of the comparison (course weeks 12 and 15: foundation models, 1-D signals).
Chronos-2 forecasts the next 34 hours of real-time price from the price history up to the **issue time** (14:00 local, D-1).
The forecast quantiles are turned into P(price ≥ spike threshold) for each hour of day D.
The probabilities are then calibrated on **val** (isotonic) and the alert threshold is tuned on val, the same as for the sklearn models.

**Before you run this**
1. Runtime → Change runtime type → **GPU** (a T4 is enough).
2. Upload the project to Google Drive as `MyDrive/AMLProject/`. It needs `src/`, `scripts/`, `configs/`, `pyproject.toml` and `data/processed/*.parquet` (`data/` is git-ignored, so copy it over too). From the project folder on Windows:
   `tar -a -cf gridwatch_colab.zip src scripts configs data pyproject.toml`, then unzip it into Drive.
3. When you're done, copy `models/chronos-*.pkl` back into the local `models/`, and compare everything locally:
   `python scripts/evaluate.py --models rule xgboost chronos-zeroshot chronos-finetuned`


In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/AMLProject'
%cd {PROJECT}
!pip install -q -e ".[chronos]"

In [ ]:
import pickle, sys
import pandas as pd
from sklearn.isotonic import IsotonicRegression

sys.path.insert(0, 'scripts')
from _common import label_col, prepare
from gridwatch.config import load_config
from gridwatch.evaluate import evaluate, pick_threshold
from gridwatch.models.chronos_model import ChronosStress

cfg = load_config()
SPEC, YCOL, EV = cfg['models']['chronos'], label_col(cfg), cfg['evaluation']
GAP = cfg['label']['event_merge_gap_hours']
data = {r: prepare(r, cfg) for r in cfg['regions']}
{r: (len(d['df']), d['price_threshold']) for r, d in data.items()}

In [ ]:
def run_and_save(name, region, model):
    """Predict on val+test, calibrate and set the threshold on val, report val metrics, save to models/{name}_{region}_{label}.pkl."""
    d = data[region]
    m, y = d['masks'], d['y'][YCOL]
    use = m['val'] | m['test']
    it = d['issue_time'][use]
    p_raw = model.predict_for_targets(d['df'][SPEC['target']], it).fillna(0.0)
    va = m['val'][use]
    iso = IsotonicRegression(out_of_bounds='clip', y_min=0, y_max=1).fit(p_raw[va], y[use][va])
    p = pd.Series(iso.predict(p_raw), index=p_raw.index)
    thr = pick_threshold(y[use][va].values, p[va].values, EV['min_recall'])
    res = evaluate(y[use][va], p[va], it[va], thr, GAP)
    print(f'[{region}] {name} val:', {k: round(v, 3) if isinstance(v, float) else v for k, v in res.items()})
    (cfg['paths']['models']).mkdir(parents=True, exist_ok=True)
    with open(cfg['paths']['models'] / f'{name}_{region}_{YCOL}.pkl', 'wb') as f:
        pickle.dump({'preds': p, 'preds_raw': p_raw, 'threshold': thr, 'price_threshold': d['price_threshold'],
                     'label': YCOL}, f)
    return p

## 1. Zero-shot
Pretrained weights, no training on grid data.

In [ ]:
zs = {}
for region, d in data.items():
    model = ChronosStress(**SPEC, threshold=d['price_threshold'])
    zs[region] = run_and_save('chronos-zeroshot', region, model)

## 2. Fine-tuned
`Chronos2Pipeline.fit` fine-tunes on the **train-split** price series only, and validation loss is computed on the val split. Test is never seen.
Settings to vary for the ablation: `FT_STEPS`, `FT_LR`, and `finetune_mode` (`'full'` or `'lora'`).

In [ ]:
FT_STEPS, FT_LR, FT_MODE = 1000, 1e-5, 'full'
ft = {}
for region, d in data.items():
    s = d['df'][SPEC['target']]
    train = s[d['masks']['train']].ffill().dropna().to_numpy('float32')
    val = s[d['masks']['val']].ffill().dropna().to_numpy('float32')
    model = ChronosStress(**SPEC, threshold=d['price_threshold'])
    model._load()
    model.pipeline = model.pipeline.fit(
        inputs=[train], validation_inputs=[val], prediction_length=34, context_length=SPEC['context_hours'],
        finetune_mode=FT_MODE, learning_rate=FT_LR, num_steps=FT_STEPS, batch_size=32,
        output_dir=f'models/chronos-ft-{region}')
    ft[region] = run_and_save('chronos-finetuned', region, model)

## 3. Quick look: zero-shot vs fine-tuned on val
The full test comparison, with significance and plots, runs locally with `scripts/evaluate.py`.

In [ ]:
from sklearn.metrics import average_precision_score
for region, d in data.items():
    va = d['masks']['val']
    y = d['y'][YCOL][va]
    print(region, {k: round(average_precision_score(y, v[region].reindex(y.index).fillna(0)), 3)
                   for k, v in [('zero-shot', zs), ('fine-tuned', ft)]})